[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [14]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [15]:
import torch
import torch.nn as nn
import math

In [16]:
# ✏️ YOUR IMPLEMENTATION HERE
# x = x + attention(layernorm(x))
# x = x + mlp(layernorm(x))

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):

      super().__init__() # inherit from nn.Module

      self.ln1 = nn.LayerNorm(d_model) # first layer norm, normalizes each token embedding before attention

      self.attn = nn.MultiheadAttention( # PyTorch built-in module, causal multi-head self-attention
          embed_dim=d_model,
          num_heads=num_heads,
          batch_first=True # means that tensors use (batch, seq_len, d_model)
      )


      self.ln2 = nn.LayerNorm(d_model) # second layer norm, before the MLP

      self.mlp = nn.Sequential( # defines feedforward network
          nn.Linear(d_model, 4 * d_model), # input layer
          nn.GELU(), # activation function, adds non-linearity
          nn.Linear(4 * d_model, d_model) # output layer
      )


    def forward(self, x):

      batch_size, seq_len, d_model = x.shape # extract dimensions for later use

      norm_x = self.ln1(x) # normalize before attention

      causal_mask = torch.triu( # runs causal self-attention
        torch.ones(seq_len, seq_len, device=x.device),
        diagonal=1
    ).bool()

      attn_out, _ = self.attn( # attention output, ignore attention weights
        norm_x,
        norm_x,
        norm_x,
        attn_mask=causal_mask
    )
      x = x + attn_out # residual connection, keeps old representation while adding attention update

      norm_x = self.ln2(x) # normalize again before MLP
      x = x + self.mlp(norm_x) # run feedforward network, add another residual connection

      return x # return transformed token representations to next Transformer block


In [17]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [18]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (5.9ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (0.9ms)
  ✅ [3/5] MLP has 4x expansion with GELU (0.9ms)
  ✅ [4/5] Causal masking — future doesn't affect past (3.2ms)
  ✅ [5/5] Gradient flow to all parameters (3.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (14.3ms total)
  Progress saved. Run status() to see your dashboard.

